# 📌 PySpark Window Functions

| Function | What it does | Example |
|----------|--------------|---------|
| `Window.partitionBy()` | Creates partitions (similar to SQL GROUP BY, but retains all rows) | `Window.partitionBy("customer_id")` |
| `Window.orderBy()` | Defines the order of rows within each partition | `.orderBy("order_date")` |
| `row_number()` | Assigns a unique sequential number to each row | 1, 2, 3, 4... |
| `rank()` | Assigns rank with gaps after ties | 1, 2, 2, 4 |
| `dense_rank()` | Assigns rank without gaps after ties | 1, 2, 2, 3 |
| `lag(col, n)` | Returns the value from `n` previous rows | Previous order amount |
| `lead(col, n)` | Returns the value from `n` next rows | Next order amount |
| `first(col)` | First value in the window | First order date |
| `last(col)` | Last value in the window | Latest order date |
| `sum().over()` | Running/Cumulative sum | Running revenue |
| `avg().over()` | Running average | Running average sales |
| `min().over()` | Running minimum | Lowest price so far |
| `max().over()` | Running maximum | Highest price so far |
| `count().over()` | Running count | Number of orders processed |
| `rowsBetween()` | Defines row-based window frame | Last 3 rows |
| `rangeBetween()` | Defines value-based window frame | Last 30 days, etc. |
| `ntile(n)` | Divides rows into `n` buckets | Quartiles, Deciles |
| `percent_rank()` | Relative rank between 0 and 1 | Ranking percentage |
| `cume_dist()` | Cumulative distribution | Percent of rows below current |

In [ ]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-16")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')

**Task 1**

Using row_number(), number all orders within each region ordered by unit_price descending. Show region, order_id, unit_price, and row_num.

In [7]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
window=Window.partitionBy('region').orderBy(F.col('unit_price').desc())
orders_df.withColumn(
    "row_num",F.row_number().over(window)
).select(
    'region','order_id','unit_price','row_num'
).show()

+------+--------+----------+-------+
|region|order_id|unit_price|row_num|
+------+--------+----------+-------+
|  East|   O0001|   1299.99|      1|
|  East|   O0051|   1299.99|      2|
|  East|   O0015|    699.99|      3|
|  East|   O0012|    599.99|      4|
|  East|   O0020|    449.99|      5|
|  East|   O0071|    449.99|      6|
|  East|   O0026|    349.99|      7|
|  East|   O0096|    349.99|      8|
|  East|   O0006|    199.99|      9|
|  East|   O0043|    199.99|     10|
|  East|   O0060|    199.99|     11|
|  East|   O0070|    199.99|     12|
|  East|   O0095|    199.99|     13|
|  East|   O0046|    109.99|     14|
|  East|   O0065|    109.99|     15|
|  East|   O0082|    109.99|     16|
|  East|   O0021|     89.99|     17|
|  East|   O0035|     89.99|     18|
|  East|   O0045|     89.99|     19|
|  East|   O0090|     89.99|     20|
+------+--------+----------+-------+
only showing top 20 rows


**Task 2**

Apply all three — row_number(), rank(), and dense_rank() — partitioned by region, ordered by unit_price descending. Show the first 10 rows and observe where the results differ.

In [15]:
window1=Window.partitionBy('region').orderBy(F.col('unit_price').desc())
orders_df.select('region',
    'order_id',
    'unit_price',
    F.row_number().over(window1).alias('row_num'),
    F.rank().over(window1).alias('rank'),
    F.dense_rank().over(window1).alias('dense_rank')
).show(10,truncate=False)

+------+--------+----------+-------+----+----------+
|region|order_id|unit_price|row_num|rank|dense_rank|
+------+--------+----------+-------+----+----------+
|East  |O0001   |1299.99   |1      |1   |1         |
|East  |O0051   |1299.99   |2      |1   |1         |
|East  |O0015   |699.99    |3      |3   |2         |
|East  |O0012   |599.99    |4      |4   |3         |
|East  |O0020   |449.99    |5      |5   |4         |
|East  |O0071   |449.99    |6      |5   |4         |
|East  |O0026   |349.99    |7      |7   |5         |
|East  |O0096   |349.99    |8      |7   |5         |
|East  |O0006   |199.99    |9      |9   |6         |
|East  |O0043   |199.99    |10     |9   |6         |
+------+--------+----------+-------+----+----------+
only showing top 10 rows


**Task 3**

Using dense_rank(), find the top 2 highest value orders per region. Filter for rank <= 2 and show the result.

In [ ]:
window3=Window.partitionBy('region').orderBy(F.col('unit_price').desc())
orders_df.select("region",'order_id','unit_price',
F.dense_rank().over(window3).alias('dense_rank')).\
        filter(F.col('dense_rank')<=2).show()

+-------+--------+----------+----------+
| region|order_id|unit_price|dense_rank|
+-------+--------+----------+----------+
|   East|   O0001|   1299.99|         1|
|   East|   O0051|   1299.99|         1|
|   East|   O0015|    699.99|         2|
|Midwest|   O0034|   1299.99|         1|
|Midwest|   O0061|   1299.99|         1|
|Midwest|   O0094|   1299.99|         1|
|Midwest|   O0098|    699.99|         2|
|  South|   O0009|   1299.99|         1|
|  South|   O0024|   1299.99|         1|
|  South|   O0056|    699.99|         2|
|   West|   O0041|   1299.99|         1|
|   West|   O0072|   1299.99|         1|
|   West|   O0080|   1299.99|         1|
|   West|   O0088|   1299.99|         1|
|   West|   O0037|    699.99|         2|
|   West|   O0078|    699.99|         2|
+-------+--------+----------+----------+



**Task 4**

For each customer, rank their orders by unit_price descending using row_number(). Filter to keep only each customer's most expensive order (row_num == 1). Join with customers.csv to show the customer's full name.

In [ ]:
joined_df=orders_df.join(customers_df,on='customer_id',how='inner')
joined_df.withColumn(
    'row_num',F.row_number().over(Window.partitionBy('customer_id').orderBy(F.col('unit_price').desc())
    )
).filter(F.col('row_num')==1).\
    select(F.concat_ws(" ",'first_name','last_name').alias('full_name')).show()

+--------------------+
|           full_name|
+--------------------+
|      James Anderson|
|        Maria Garcia|
|      Robert Johnson|
|      Linda Martinez|
|       Michael Brown|
|      Patricia Davis|
|      William Wilson|
|      Barbara Taylor|
|        David Thomas|
|    Jennifer Jackson|
|       Richard White|
|        Susan Harris|
|       Joseph Martin|
|    Jessica Thompson|
|       Thomas Garcia|
|      Sarah Martinez|
|    Charles Robinson|
|         Karen Clark|
|Christopher Rodri...|
|         Nancy Lewis|
+--------------------+
only showing top 20 rows


26/07/31 14:03:47 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 2024862 ms exceeds timeout 120000 ms
26/07/31 14:03:47 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/31 14:03:48 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at 